# Arbeitspaket (AP) 1: Management & Nutzung relationaler Daten — Probeprüfung 2 (Fitnessstudio) · MUSTERLÖSUNG

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung &amp; Design</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (fitness.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (fitness.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `fitness.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `courses`, `studios`, `trainers`, `sessions`, `members`, `bookings`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `fitness_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [1]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /home/claude/out


### Datenbankverbindung zu PostgreSQL

In [2]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

Host:       localhost
Datenbank:  postgres


### Daten nach PostgreSQL laden

In [3]:
# Tabellen aus der SQLite-Datei fitness.db nach PostgreSQL übertragen
tables = ['courses', 'studios', 'trainers', 'sessions', 'members', 'bookings']

sqlite_engine = create_engine('sqlite:///fitness.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

courses        12 Zeilen
studios         6 Zeilen
trainers       30 Zeilen
sessions      300 Zeilen


members       505 Zeilen


bookings     2000 Zeilen


> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `fitness.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///fitness.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `fitness.db` werden nicht gewertet.*

### A1 (4 Punkte)

Die Tabelle `trainers` enthält `studioid` (Heimstudio der Trainerin / des Trainers) und die Tabelle `sessions` enthält ebenfalls `studioid` sowie `trainerid`.

Ein Datenbankarchitekt sagt: *«`sessions.studioid` ist redundant — das Studio lässt sich über `sessions.trainerid` → `trainers.studioid` ermitteln. Die Spalte sollte entfernt werden.»*

Hat er recht? Begründen Sie Ihre Antwort mit einem konkreten Szenario aus dem Studiobetrieb (z.B. Vertretungen, Studiowechsel).

**Ihre Antwort:**

> **Nein.** `trainers.studioid` beschreibt das *aktuelle Heimstudio* einer Person, `sessions.studioid` den *Ort eines konkreten Kurstermins*. Das sind zwei verschiedene Fakten.
> 
> - **Vertretung:** Eine Trainerin mit Heimstudio Zürich (ST01) springt in Winterthur (ST06) ein. Ohne `sessions.studioid` würde der Termin fälschlich Zürich zugeordnet. In `fitness.db` finden 33 der 300 Termine nicht im Heimstudio der Trainer:in statt.
> - **Studiowechsel:** Wechselt ein Trainer 2026 von Bern nach Luzern, wird `trainers.studioid` überschrieben. Alle früheren Termine würden rückwirkend Luzern zugeordnet, Buchungen und Umsatz pro Studio wären historisch falsch.
> 
> Das Studio ist also ein Attribut des Termins, nicht nur des Trainers, und muss in `sessions` bleiben.

### A2 (4 Punkte)

Nach einer Systemmigration stellt das Datenteam fest, dass bei 180 der 2000 Buchungen in `bookings` die Spalte `paid_price` leer (`NULL`) ist. Alle anderen Spalten dieser Buchungen sind befüllt.

Beschreiben Sie das konkrete Datenproblem und welche Auswirkung es auf eine Umsatzauswertung pro Studio hat. Nennen Sie die passende Data-Quality-Eigenschaft aus den Vorlesungsfolien (z.B. accurate, consistent, complete, reliable).

**Ihre Antwort:**

> **Problem:** Für 180 Buchungen (9 %) fehlt der bezahlte Preis. Die Buchung existiert, aber der Betrag ist unbekannt. `NULL` ist dabei nicht dasselbe wie CHF 0 (z.B. Gratis-Probetraining).
> 
> **Auswirkung:** `SUM(paid_price)` ignoriert `NULL`-Werte, der Umsatz pro Studio wird also systematisch **unterschätzt**. Sind die fehlenden Werte ungleich verteilt (z.B. nur Buchungen eines Studios oder nur Gutscheinbuchungen betroffen), ist auch der Vergleich zwischen Studios und `AVG(paid_price)` verzerrt.
> 
> **Data-Quality-Eigenschaft:** **complete** (Vollständigkeit).

### A3 (4 Punkte)

In `fitness.db` hat kein Kurstermin mehr Buchungen als `max_participants` des zugehörigen Kurses. Angenommen, ein Termin von «Pilates Reformer» (`max_participants` = 10) hätte 12 Buchungen: Wäre das **zwingend** ein Datenfehler? Begründen Sie in einem Satz.

**Ihre Antwort:**

> **Nein, nicht zwingend:** `bookings` hat keine Status-Spalte, die 12 Zeilen könnten also auch stornierte Buchungen oder Wartelisten-Einträge enthalten, oder das Studio überbucht bewusst wegen erwarteter No-Shows; ein Fehler wäre es erst, wenn tatsächlich mehr als 10 Personen teilgenommen hätten.

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `fitness.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Anzahl Buchungen pro Kurs — für **alle** Kurse in der Datenbank.

```sql
SELECT c.title, COUNT(*) AS booking_count
FROM sessions s
JOIN bookings b ON s.sessionid = b.sessionid
JOIN courses c ON s.courseid = c.courseid
GROUP BY c.title;
```

Die Tabelle `courses` enthält **12 Kurse**. Tatsächliches Ergebnis der SQL-Abfrage: nur **11 Zeilen**, Buchungsanzahl zwischen 113 und 275.

Die Studioleitung fragt: *«Zeig mir auch Kurse, die noch gar nicht gebucht wurden.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 12 Kurse** erscheinen und ein Kurs ohne Buchungen korrekt mit `0` ausgewiesen wird. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> 1. **`courses` als Ausgangstabelle (FROM):** Es sollen *alle Kurse* erscheinen, also muss die Tabelle, deren Zeilen vollständig erhalten bleiben sollen, links stehen.
> 2. **`LEFT JOIN` statt `JOIN` (beide Joins):** Der `INNER JOIN` verwirft «Aqua Fit», weil es dazu keine Kurstermine gibt. Mit `LEFT JOIN` bleibt der Kurs erhalten, die Spalten aus `sessions`/`bookings` sind `NULL`. Beide Joins müssen `LEFT` sein, sonst entfernt der zweite `INNER JOIN` die Zeile wieder.
> 3. **`COUNT(b.bookingid)` statt `COUNT(*)`:** `COUNT(*)` zählt Zeilen, also auch die `NULL`-Zeile von «Aqua Fit» (Ergebnis 1 statt 0). `COUNT(spalte)` zählt nur Nicht-`NULL`-Werte. Zusätzlich gibt es Kurstermine ohne Buchung: Mit `COUNT(*)` hätte «Tabata» 157 statt 156 und «Pilates Mat» 190 statt 189.
> 4. **`ORDER BY` (optional):** sortiert das Ergebnis lesbar, z.B. absteigend nach Anzahl.

**Korrigierte SQL-Abfrage:**

In [4]:
engine = create_engine(db_connection_url)

df = pd.read_sql_query("""
SELECT c.title, COUNT(b.bookingid) AS booking_count
FROM courses c
LEFT JOIN sessions s ON c.courseid = s.courseid
LEFT JOIN bookings b ON s.sessionid = b.sessionid
GROUP BY c.title
ORDER BY booking_count DESC;
""", con=engine)

engine.dispose()
df

,title,booking_count
0,Power Yoga,275
1,Boxfit,235
2,Yin Yoga,202
3,Functional Training,201
4,Pilates Mat,189
5,Spinning Intervall,173
6,HIIT Express,160
7,Tabata,156
8,Spinning Classic,152
9,Kettlebell,144


### B2 (8 Punkte)

**Ziel:** Liste aller Trainer:innen (`trainerid`, `firstname`, `lastname`), die noch nie einen Kurstermin geleitet haben.

```sql
SELECT t.trainerid, t.firstname, t.lastname
FROM trainers t
LEFT JOIN sessions s ON t.trainerid = s.trainerid
WHERE s.sessionid = NULL;
```

Tatsächliches Ergebnis: 0 Zeilen — obwohl kürzlich neue Trainer:innen eingestellt wurden, die noch keine Kurse geleitet haben.

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, die alle Trainer:innen ohne Kurstermin ausgibt. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> 1. **`IS NULL` statt `= NULL`:** `NULL` bedeutet «unbekannt». Jeder Vergleich mit `=` ergibt deshalb `NULL` (nicht `TRUE`), auch `NULL = NULL`. Die `WHERE`-Bedingung ist nie erfüllt, darum 0 Zeilen. Auf `NULL` prüft man ausschliesslich mit `IS NULL` / `IS NOT NULL`.
> 2. **`LEFT JOIN` bleibt:** Der Join ist korrekt. Er liefert für Trainer:innen ohne Termin eine Zeile mit `s.sessionid = NULL`, genau diese Zeilen filtert die korrigierte Bedingung heraus (Anti-Join).
> 3. **`ORDER BY` (optional)** für eine lesbare Ausgabe.
>
> Alternative, ebenfalls korrekt: `WHERE NOT EXISTS (SELECT 1 FROM sessions s WHERE s.trainerid = t.trainerid)`.
>
> Ergebnis: 4 Trainer:innen (T026–T029), alle 2026 eingestellt.

**Korrigierte SQL-Abfrage:**

In [5]:
engine = create_engine(db_connection_url)

df = pd.read_sql_query("""
SELECT t.trainerid, t.firstname, t.lastname
FROM trainers t
LEFT JOIN sessions s ON t.trainerid = s.trainerid
WHERE s.sessionid IS NULL
ORDER BY t.trainerid;
""", con=engine)

engine.dispose()
df

,trainerid,firstname,lastname
0,T026,Laura,Berger
1,T027,Mia,Steiner
2,T028,Simon,Keller
3,T029,Julia,Weber


---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `fitness.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Die gültigen Werte in `members.membership_type` sind `Basic`, `Premium`, `Student` und `Firmenabo`. Schreiben Sie eine SQL-Abfrage, die alle Mitglieder (`memberid`, `firstname`, `lastname`, `membership_type`) mit einem ungültigen Wert in `membership_type` ausgibt — d.h. alle Werte ausser diesen vier. Begründen Sie Ihre Filterbedingung.

**Begründung der Vorgehensweise:**

> `NOT IN` mit der Liste der vier gültigen Werte gibt jede Abweichung zurück, ohne dass man die Fehlervarianten kennen muss. Der Vergleich ist in PostgreSQL und SQLite **case-sensitive**, deshalb werden auch `basic` und `BASIC` erkannt; mit `LOWER()` oder `ILIKE` würden diese fälschlich als gültig durchgehen. (Hinweis: `NULL`-Werte würden von `NOT IN` nicht erfasst und bräuchten `OR membership_type IS NULL`; in `fitness.db` gibt es keine.)

**SQL-Abfrage:**

In [6]:
engine = create_engine(db_connection_url)

df = pd.read_sql_query("""
SELECT memberid, firstname, lastname, membership_type
FROM members
WHERE membership_type NOT IN ('Basic', 'Premium', 'Student', 'Firmenabo')
ORDER BY memberid;
""", con=engine)

engine.dispose()
df

,memberid,firstname,lastname,membership_type
0,M0501,Laura,Frei,basic
1,M0502,Jan,Graf,BASIC
2,M0503,Nina,Steiner,Basik
3,M0504,Paul,Meier,premium
4,M0505,Lea,Huber,Firmen-Abo


### C2 (4 Punkte)

Wie hoch ist der durchschnittliche bezahlte Preis (`paid_price`) pro Kurskategorie (`category`)? Geben Sie `category` und den Durchschnittspreis (`avg_price`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_price`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> `paid_price` steht in `bookings`, `category` in `courses`. Es gibt keine direkte Beziehung zwischen den beiden, deshalb muss über `sessions` verbunden werden (`bookings.sessionid` → `sessions.courseid` → `courses`). Ein `INNER JOIN` genügt, da Kategorien ohne Buchungen keinen Durchschnitt haben.

**SQL-Abfrage:**

In [7]:
engine = create_engine(db_connection_url)

df = pd.read_sql_query("""
SELECT c.category,
       ROUND(CAST(AVG(b.paid_price) AS NUMERIC), 2) AS avg_price
FROM bookings b
JOIN sessions s ON b.sessionid = s.sessionid
JOIN courses c  ON s.courseid  = c.courseid
GROUP BY c.category
ORDER BY avg_price DESC;
""", con=engine)

engine.dispose()
df

,category,avg_price
0,Pilates,27.39
1,Boxen,26.67
2,Krafttraining,22.77
3,Yoga,21.14
4,Spinning,19.97
5,HIIT,17.50


### C3 (4 Punkte)

Welche Studios haben einen Umsatz (Summe aller `paid_price`) von mehr als CHF 7000 erzielt? Geben Sie `name` und den Umsatz (`revenue`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`, absteigend sortiert nach `revenue`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> `name` steht in `studios`, `paid_price` in `bookings`. Die Verbindung läuft über `sessions`, weil eine Buchung einem Kurstermin gehört und erst der Termin das Studio kennt (`sessions.studioid`, nicht `trainers.studioid`, vgl. A1). Die Bedingung «Umsatz > 7000» bezieht sich auf ein Aggregat und gehört deshalb in `HAVING`, nicht in `WHERE`.

**SQL-Abfrage:**

In [8]:
engine = create_engine(db_connection_url)

df = pd.read_sql_query("""
SELECT st.name,
       ROUND(CAST(SUM(b.paid_price) AS NUMERIC), 2) AS revenue
FROM studios st
JOIN sessions s ON st.studioid = s.studioid
JOIN bookings b ON s.sessionid = b.sessionid
GROUP BY st.name
HAVING SUM(b.paid_price) > 7000
ORDER BY revenue DESC;
""", con=engine)

engine.dispose()
df

,name,revenue
0,PulsWerk Oerlikon,12121.9
1,Aktiv Bern,8656.5
2,FitWerk Winterthur,7209.1


---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [9]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.18.44-fc-v242
Datetime: 2026-11-06 20:20:55
Python Version: 3.13.18
IP Address: 127.0.0.1
-----------------------------------
